# GEO-ADAPT Wildfire Summer School 2026
## 00 — Environment preflight

Run **Run → Run All Cells** before the course.

The notebook supports both:

- **CDSE JupyterLab** — the primary course environment;
- **local Python 3.11** — the supported fallback environment.

At the end, the target message is **YOUR COURSE ENVIRONMENT IS READY**.

- `FAIL` = a blocker for the core practicals.
- `WARN` = an optional service or extension is unavailable.

If a `FAIL` remains after following the setup instructions, send the complete error text or a screenshot of the final summary to the trainers.

## 1. Earth Engine configuration

The course does **not** hard-code a trainer's Google Cloud project.

If Earth Engine can infer a project from your existing credentials, leave `GEE_PROJECT_ID` empty. If your account requires an explicit registered Google Cloud project, set the environment variable before starting Jupyter or enter the project ID below.

In [ ]:
import os

GEE_PROJECT_ID = os.environ.get("GEE_PROJECT_ID", "").strip()

# Optional local override for this notebook only:
# GEE_PROJECT_ID = "your-google-cloud-project-id"

# Optional NASA FIRMS API key. Not required for the six core notebooks.
# Prefer an environment variable rather than pasting credentials into notebooks.
FIRMS_MAP_KEY = os.environ.get("FIRMS_MAP_KEY", "").strip()

print("GEE project:", GEE_PROJECT_ID or "<not explicitly set>")

## 2. Preflight helpers

In [ ]:
from __future__ import annotations

import importlib
import platform
import shutil
import subprocess
import sys
from pathlib import Path

RESULTS = []

def record(name, status, detail=""):
    status = status.upper()
    RESULTS.append((name, status, str(detail)))
    icon = {"PASS": "✓", "WARN": "!", "FAIL": "✗"}.get(status, "?")
    print(f"{icon} {status:<4} | {name}" + (f" — {detail}" if detail else ""))

def safe_test(name, fn, optional=False):
    try:
        detail = fn()
        record(name, "PASS", detail or "")
        return True
    except Exception as exc:
        record(
            name,
            "WARN" if optional else "FAIL",
            f"{type(exc).__name__}: {exc}",
        )
        return False

def import_test(import_name, display_name=None, optional=False):
    display_name = display_name or import_name
    try:
        module = importlib.import_module(import_name)
        version = getattr(module, "__version__", "version unavailable")
        record(display_name, "PASS", version)
        return module
    except Exception as exc:
        record(
            display_name,
            "WARN" if optional else "FAIL",
            f"{type(exc).__name__}: {exc}",
        )
        return None

def find_repo_root():
    candidates = [
        Path.cwd(),
        Path.cwd().parent,
        Path.home() / "mystorage" / "fire-school",
    ]
    for candidate in candidates:
        if (candidate / "data").exists() and (candidate / "notebooks").exists():
            return candidate
    raise FileNotFoundError(
        "Could not find the fire-school repository. "
        "Open this notebook from the cloned repository."
    )

REPO_ROOT = find_repo_root()
print("Repository:", REPO_ROOT)

## 3. Python and core package checks

The canonical local interpreter is **Python 3.11**. CDSE may update its managed environment over time; if CDSE is not on 3.11 but all core imports work, ask a trainer before changing anything.

In [ ]:
if sys.version_info[:2] == (3, 11):
    record("Python 3.11", "PASS", sys.version.split()[0])
else:
    # CDSE is managed by the platform; a different version there may still be
    # usable, while the supported local fallback remains Python 3.11.
    in_cdse = (Path.home() / "mystorage").exists()
    record(
        "Python 3.11",
        "WARN" if in_cdse else "FAIL",
        f"Current interpreter: {sys.version.split()[0]}",
    )

record("Operating system", "PASS", f"{platform.system()} {platform.release()}")

np = import_test("numpy", "NumPy")
pd = import_test("pandas", "Pandas")
gpd = import_test("geopandas", "GeoPandas")
shapely = import_test("shapely", "Shapely")
folium = import_test("folium", "Folium")
xr = import_test("xarray", "xarray")
netcdf4 = import_test("netCDF4", "netCDF4")
mpl = import_test("matplotlib", "Matplotlib")
requests = import_test("requests", "Requests")
pystac_client = import_test("pystac_client", "pystac-client", optional=True)
openeo = import_test("openeo", "openEO client", optional=True)
ee = import_test("ee", "Earth Engine Python API")

# Optional extensions only.
rasterio = import_test("rasterio", "Rasterio", optional=True)
rioxarray = import_test("rioxarray", "rioxarray", optional=True)
sklearn = import_test("sklearn", "scikit-learn", optional=True)
geemap = import_test("geemap", "geemap", optional=True)

### If a core package is missing

For a local installation, activate the course environment and run:

```bash
python -m pip install -r requirements-lock.txt
```

In CDSE JupyterLab, do not install packages into the managed environment unless a trainer asks you to.

## 4. Basic geospatial and plotting checks

In [ ]:
def test_dataframe():
    import pandas as pd
    df = pd.DataFrame({"site": ["A", "B"], "value": [1, 2]})
    assert int(df["value"].sum()) == 3
    return "DataFrame operations work"

safe_test("Pandas calculation", test_dataframe)

def test_geodataframe():
    import geopandas as gpd
    from shapely.geometry import Point

    gdf = gpd.GeoDataFrame(
        {"site": ["Ohrid"]},
        geometry=[Point(20.8016, 41.1231)],
        crs="EPSG:4326",
    )
    assert len(gdf) == 1
    assert str(gdf.crs) == "EPSG:4326"
    return "GeoDataFrame + CRS work"

safe_test("GeoPandas / Shapely", test_geodataframe)

def test_matplotlib():
    import matplotlib.pyplot as plt

    fig, ax = plt.subplots(figsize=(4, 2.4))
    ax.plot([0, 1, 2], [0, 1, 0])
    ax.set_title("Preflight plot")
    plt.show()
    return "Plot rendered"

safe_test("Matplotlib rendering", test_matplotlib)

## 5. Repository, Git and writable storage

In [ ]:
def test_git():
    if shutil.which("git") is None:
        raise RuntimeError("git executable not found")
    version = subprocess.run(
        ["git", "--version"],
        check=True,
        capture_output=True,
        text=True,
    ).stdout.strip()
    return version

safe_test("Git", test_git)

def test_storage():
    mystorage = Path.home() / "mystorage"
    target_dir = mystorage if mystorage.exists() else REPO_ROOT

    probe = target_dir / "_geo_adapt_preflight_test.txt"
    probe.write_text("GEO-ADAPT preflight OK\n", encoding="utf-8")
    text = probe.read_text(encoding="utf-8").strip()
    probe.unlink()
    assert text == "GEO-ADAPT preflight OK"

    mode = "CDSE persistent mystorage" if mystorage.exists() else "local repository"
    return f"Read/write/delete works in {mode}: {target_dir}"

safe_test("Writable course storage", test_storage)

## 6. Required course data

These files are committed to the repository. Students do not need CDS credentials or a separate Google Drive download.

In [ ]:
REQUIRED_DATA = [
    Path("data/aoi/galicica_aoi.geojson"),
    Path("data/effis/Galicica.gpkg"),
    Path("data/weather/era5_land_galicica_hourly_2024.nc"),
    Path("data/weather/era5_land_galicica_fireseason_1991_latest.nc"),
]

def test_course_files():
    missing = []
    for relative in REQUIRED_DATA:
        path = REPO_ROOT / relative
        if not path.exists() or path.stat().st_size == 0:
            missing.append(str(relative))
    if missing:
        raise FileNotFoundError("Missing: " + ", ".join(missing))
    return "AOI, EFFIS and both ERA5-Land datasets present"

safe_test("Committed course data", test_course_files)

def test_netcdf_course_files():
    import xarray as xr

    files = [
        REPO_ROOT / "data/weather/era5_land_galicica_hourly_2024.nc",
        REPO_ROOT / "data/weather/era5_land_galicica_fireseason_1991_latest.nc",
    ]

    details = []
    for path in files:
        with xr.open_dataset(path, engine="netcdf4") as ds:
            if "time" not in ds.coords or ds.time.size == 0:
                raise RuntimeError(f"{path.name}: no usable time coordinate")
            details.append(f"{path.name}: {ds.time.size} time steps")
    return "; ".join(details)

safe_test("NetCDF4 course datasets", test_netcdf_course_files)

## 7. Internet connectivity

In [ ]:
def test_internet():
    import requests

    response = requests.get("https://dataspace.copernicus.eu", timeout=15)
    response.raise_for_status()
    return f"CDSE website HTTP {response.status_code}"

safe_test("Internet / CDSE website", test_internet)

## 8. CDSE openEO backend — optional service check

The current core notebooks do not depend on openEO, so an outage here is a warning rather than a blocker.

In [ ]:
def test_openeo_backend():
    import openeo

    connection = openeo.connect("openeo.dataspace.copernicus.eu")
    capabilities = connection.capabilities()
    version = getattr(capabilities, "api_version", None)
    return "Backend reachable" + (f"; API {version}" if version else "")

safe_test("CDSE openEO backend", test_openeo_backend, optional=True)

## 9. Public STAC access — optional service check

In [ ]:
def test_stac():
    from pystac_client import Client

    catalog = Client.open("https://earth-search.aws.element84.com/v1")
    search = catalog.search(
        collections=["sentinel-2-l2a"],
        bbox=[20.6, 40.9, 21.1, 41.4],
        datetime="2024-07-01/2024-07-10",
        max_items=1,
    )
    items = list(search.items())
    if not items:
        raise RuntimeError("STAC endpoint responded but returned no test item.")
    return f"STAC reachable; example item: {items[0].id}"

safe_test("Public STAC catalogue", test_stac, optional=True)

## 10. Google Earth Engine

Earth Engine is required for the Sentinel-2 and susceptibility practicals.

The cell first tries existing credentials. If they are unavailable, it starts the standard authentication flow. If authentication succeeds but initialization still fails, the most common reason is that the account needs an explicit registered Google Cloud project.

In [ ]:
def initialize_ee():
    import ee

    def do_initialize():
        if GEE_PROJECT_ID:
            ee.Initialize(project=GEE_PROJECT_ID)
        else:
            ee.Initialize()

    try:
        do_initialize()
    except Exception:
        print("Existing Earth Engine credentials were not usable.")
        print("Starting Earth Engine authentication...")
        ee.Authenticate()
        try:
            do_initialize()
        except Exception as exc:
            if not GEE_PROJECT_ID:
                raise RuntimeError(
                    "Earth Engine authentication completed, but no usable "
                    "Google Cloud project was found. Set GEE_PROJECT_ID to a "
                    "registered project and rerun this cell."
                ) from exc
            raise

    value = ee.Number(10).pow(2).multiply(3).subtract(5).getInfo()
    if value != 295:
        raise RuntimeError(f"Unexpected Earth Engine result: {value}")

    bands = ee.Image("USGS/SRTMGL1_003").bandNames().getInfo()
    if "elevation" not in bands:
        raise RuntimeError(f"Unexpected SRTM bands: {bands}")

    project_text = GEE_PROJECT_ID or "project inferred from credentials"
    return f"{project_text}; compute + SRTM catalogue access OK"

safe_test("Google Earth Engine", initialize_ee)

## 11. Optional NASA FIRMS API

A FIRMS key is **not required** for the six core notebooks. This test is included for participants using the near-real-time extension.

In [ ]:
def test_firms():
    import requests

    if not FIRMS_MAP_KEY:
        raise RuntimeError("No FIRMS_MAP_KEY configured — live test skipped.")

    url = (
        "https://firms.modaps.eosdis.nasa.gov/api/area/csv/"
        f"{FIRMS_MAP_KEY}/VIIRS_SNPP_NRT/20.5,40.8,21.2,41.5/1"
    )
    response = requests.get(url, timeout=20)
    response.raise_for_status()

    if "latitude" not in response.text.lower():
        raise RuntimeError("FIRMS response did not look like expected CSV.")
    return "FIRMS API reachable"

safe_test("NASA FIRMS API", test_firms, optional=True)

## 12. Final readiness summary

In [ ]:
from collections import Counter

counts = Counter(status for _, status, _ in RESULTS)

print("\n" + "=" * 72)
print("PREFLIGHT SUMMARY")
print("=" * 72)
print(f"PASS: {counts.get('PASS', 0)}")
print(f"WARN: {counts.get('WARN', 0)}")
print(f"FAIL: {counts.get('FAIL', 0)}")

fails = [(name, detail) for name, status, detail in RESULTS if status == "FAIL"]
warns = [(name, detail) for name, status, detail in RESULTS if status == "WARN"]

if fails:
    print("\nBLOCKERS:")
    for name, detail in fails:
        print(f"  ✗ {name}: {detail}")
    print("\n" + "!" * 72)
    print("YOUR COURSE ENVIRONMENT IS NOT READY YET")
    print("Send the FAIL messages above to the trainers if the setup guide does not resolve them.")
    print("!" * 72)
else:
    print("\n" + "=" * 72)
    print("YOUR COURSE ENVIRONMENT IS READY")
    print("=" * 72)
    if warns:
        print("\nOptional warnings:")
        for name, detail in warns:
            print(f"  ! {name}: {detail}")

### If something fails

Before contacting the trainers:

1. confirm that the repository is up to date with `git pull`;
2. for a local environment, run `python scripts/check_environment.py`;
3. confirm that the selected Jupyter kernel is the intended course environment.

Then send:

- a screenshot of the **PREFLIGHT SUMMARY**;
- the complete failed-check message;
- whether you are using **CDSE JupyterLab** or the **local Python 3.11 fallback**.